# Notebook 1 — Setup

This notebook introduces the tooling and libraries for learning Pong from pixels.

The figures below are explanatory only — you do not need to run any code to see them.
Run only the code cells (imports and setup) when you want to try the APIs yourself.


## Running this notebook

**Locally**: the environment comes from the repo. `uv sync --locked`, then
`uv run jupyter lab`. The cell below does nothing.

**On Colab**: the kernel exists before the project does, so the cell below clones the
repo and installs what Colab is missing. Run it first.

It installs `gymnasium` and `ale-py` only. Colab already provides a working CUDA build
of torch, and replacing it with the exact locked one means downloading the entire CUDA
stack for no benefit to anything taught here. The README describes the stricter route
if you ever need bit-identical versions.


In [ ]:
# Colab bootstrap. Does nothing when you run this notebook locally.
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # The -d test makes re-running this cell harmless.
    ![ -d /content/pong-pytorch ] || git clone -q https://github.com/piyushahuja/pong-pytorch.git /content/pong-pytorch
    %cd /content/pong-pytorch
    !curl -LsSf https://astral.sh/uv/install.sh | sh
    !~/.local/bin/uv pip install -q --system --python {sys.executable} gymnasium ale-py
    !~/.local/bin/uv pip install -q --system --python {sys.executable} -e . --no-deps

print("python:", sys.version.split()[0])
if IN_COLAB:
    print("working directory:", os.getcwd())


In [ ]:
# Confirm the environment is what you think it is.
import torch
import gymnasium
import ale_py

print("torch:     ", torch.__version__)
print("gymnasium: ", gymnasium.__version__)
print("ale-py:    ", ale_py.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Project setup

There is nothing to create: the repo *is* the project. From a clone,

```bash
uv sync --locked
```

provisions Python, installs the exact versions in `uv.lock`, and installs the `pong`
package itself in editable mode — which is what makes `import pong` work in these
notebooks without any `sys.path` juggling. Then use the project `.venv` as your
notebook kernel.

`--locked` fails rather than quietly re-resolving if `pyproject.toml` and `uv.lock` have
drifted, so everyone gets the same environment. Use it every time; there is no reason to
run plain `uv sync` here.

Do not run `uv init` in a clone — it would overwrite `pyproject.toml`. That command is
for starting a project from nothing, which is not what you are doing.

To add a dependency later, `uv add <package>`. It changes two files, `pyproject.toml`
and `uv.lock`; commit both together, or `uv sync --locked` breaks for everyone else.


## Same policy API, different environments

A PyTorch policy talks to the world through Gymnasium. Under that API you can plug in MuJoCo (robots) or ALE (Atari Pong). The learning loop shape stays the same.

![PyTorch policy stacks for MuJoCo vs ALE Pong](../assets/stack-policy-envs.svg)

## ALE (`ale-py`)

ALE = Arcade Learning Environment. The `ale-py` package lets Gymnasium run Atari 2600 games like Pong.

It is an emulator-based research environment: it runs the actual game ROM, keeps internal state, accepts controller actions, and returns pixels and rewards.

ALE does not know about neural networks. It only simulates Atari:

$$
(\text{game state},\, \text{action})
\rightarrow
(\text{new state},\, \text{pixels},\, \text{reward})
$$

In [ ]:
import ale_py

## Gymnasium

Gymnasium is the RL API that connects an agent to an environment. The two calls you will use constantly:

```python
obs, info = env.reset()
obs, reward, terminated, truncated, info = env.step(action)
```

In [ ]:
import gymnasium as gym

## PyTorch (`torch`)

PyTorch is the deep-learning library for this project. You will use it to:

- define a policy network (`torch.nn`)
- run a forward pass from pixels to action probabilities
- compute a loss, call `.backward()`, and update weights with an optimizer

It does not simulate Pong; Gymnasium + ALE do that. Torch only owns the learnable policy and gradients.

In [ ]:
import torch
import torch.nn as nn

## Small utilities

These show up later for math helpers and saving checkpoints.

In [ ]:
import math  # e.g. sqrt
from pathlib import Path  # save checkpoints